# Tutorial - Static and Modal Analyses

This is the second part of the ROSS tutorial. It shows how to run static analysis, modal analysis, and the Campbell diagram on a `Rotor`.

Each analysis starts with a `.run_*` method. These methods return a results object with plot methods that use [Plotly](https://plotly.com/python/) — for example `run_campbell().plot()` and `run_modal().plot_mode_3d(mode)`. You can store and customize the figures through the Plotly API.

Plot methods accept Pint units, see [Tutorial - Units](tutorial_units.ipynb).


In [1]:
import ross as rs
import numpy as np

# Make sure the default renderer is set to 'notebook_connected' for inline plots in Jupyter
import plotly.io as pio

pio.renderers.default = "notebook_connected"

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Rotor model

Building the model is covered in [Tutorial - Assembling a Rotor](tutorial_modeling_part_2.ipynb). However, the analyses below use the built-in compressor example.


In [2]:
rotor3 = rs.compressor_example()

node_increment = 5
rotor3.plot_rotor(nodes=node_increment)

# Section 1: Static Analysis

This method runs the static analysis for the rotor. It calculates the static deformation due to the gravity effects (shaft and disks weight). It also returns the bending moment and shearing force on each node, and you can return a free-body-diagram representation for the rotor, with the self weight, disks weight and reaction forces on bearings displayed.

## 1.1 Running static analysis

To run the simulation, use the `.run_static()` method. You can define a variable to store the results.

Storing the results, it's possible to return the following arrays:
- `disk_forces_nodal`

- `disk_forces_tag`

- `bearing_forces_nodal`

- `bearing_forces_tag`

- `disp_y`

- `Vx`

- `Bm`



In [3]:
static = rotor3.run_static()

### 1.1.1 Returning disk forces
- `.disk_forces_nodal`:
Returns a dictionary specifying the node where the disk is located and the force value.

- `.disk_forces_tag`:
Returns a dictionary mapping each disk tag to its force value and the force value.

### 1.1.2 Returning bearing forces
- `.bearing_forces_nodal`:
Returns a dictionary specifying the node where the bearing is located and the force value.

- `.bearing_forces_tag`:
Returns a dictionary mapping each bearing tag to its force value and the force value.

In [4]:
print("Disk forces - nodes")
print(rotor3.disk_forces_nodal)
print("")
print("Disk forces - tags")
print(rotor3.disk_forces_tag)

print("")
print("Bearing forces - nodes")
print(rotor3.bearing_forces_nodal)
print("")
print("Bearing forces - tags")
print(rotor3.bearing_forces_tag)

Disk forces - nodes
{'node_3': 136.408415, 'node_20': 65.801615, 'node_23': 67.66485, 'node_26': 66.29194, 'node_29': 66.586135, 'node_32': 67.17452499999999, 'node_35': 68.05711000000001}

Disk forces - tags
{'Disk 0': 136.408415, 'Disk 1': 65.801615, 'Disk 2': 67.66485, 'Disk 3': 66.29194, 'Disk 4': 66.586135, 'Disk 5': 67.17452499999999, 'Disk 6': 68.05711000000001}

Bearing forces - nodes
{'node_7': np.float64(1179.8283315153449), 'node_48': np.float64(1181.5991789515167)}

Bearing forces - tags
{'Bearing 0': np.float64(1179.8283315153449), 'Bearing 1': np.float64(1181.5991789515167)}


### 1.1.3 Other attributes

- `.Vx`: Shearing force array.

- `.Bm`: Bending moment array.

- `.deformation` Displacement in Y direction.


## 1.2 Plotting results 

With results stored, you can use some methods to plot the results. Currently, there're four plots you can retrieve from static analysis:

- `.plot_free_body_diagram()`

- `.plot_deformation()`

- `.plot_shearing_force()`

- `.plot_bending_moment()`


### 1.2.1 Plotting free-body-diagram

In [5]:
static.plot_free_body_diagram()

### 1.2.2 Plotting deformation

In [6]:
static.plot_deformation()

### 1.2.3 Plotting shearing force diagram

In [7]:
static.plot_shearing_force()

### 1.2.4 Plotting bending moment diagram

In [8]:
static.plot_bending_moment()

# Section 2: Modal Analysis

ROSS performs the modal analysis through method `run_modal()`.
This method calculates natural frequencies, damping ratios and mode shapes.

You must select a speed, which will be used as excitation frequency to calculate the system's eigenvalues and eigenvectors, and the number of eigenvalues and eigenvectors to be calculated is an optional argument (`num_modes`).

After running the modal analysis, it's possible to return the following attributes:

- eigenvalues (`evalues`);

- eigenvectors (`evectors`);

- damped natural frequencies (`wd`);

- undamped natural frequencies (`wn`);

- damping ratio (`damping_ratio`);

- logarithmic decrement (`log_dec`).

## 2.1 Running modal analysis

To run the modal analysis, choose a speed to instantiate the method. For different speeds, change the argument and run `run_modal()` once again.

### 2.1.1 Returning undamped natural frequencies

In [9]:
rotor_speed = 100.0  # rad/s
modal = rotor3.run_modal(rotor_speed, num_modes=16)
print(f"Undamped natural frequencies:\n {modal.wn}")

Undamped natural frequencies:
 [ 969.2901354   982.49764627 2014.74831088 2028.32700247 3082.12686836
 3141.86998369 3140.49185782 4506.68321354]


### 2.1.2 Returning damped natural frequencies

In [10]:
# modal.wd
print(f"Damped natural frequencies:\n {modal.wd}")

Damped natural frequencies:
 [ 952.22400319  966.42579235 2001.64160167 2015.19949502 3044.21931002
 3103.76758902 3140.49185782 4269.69780714]


### 2.1.3 Returning the damping ratio

In [11]:
print(f"Damping ratio for each mode:\n {modal.damping_ratio}")

Damping ratio for each mode:
 [ 1.86825241e-01  1.80135313e-01  1.13879084e-01  1.13588225e-01
  1.56355500e-01  1.55265983e-01 -1.43821926e-13  3.20008493e-01]


### 2.1.4 Returning logarithmic decrement 

In [12]:
print(f"Logarithmic decrement for each mode:\n {modal.log_dec}")

Logarithmic decrement for each mode:
 [ 1.19489595e+00  1.15064600e+00  7.20208621e-01  7.18345055e-01
  9.94643857e-01  9.87541146e-01 -9.03659815e-13  2.12227308e+00]


## 2.2 Plotting results

Once `run_modal()` is completed, you can check for the rotor's mode shapes. You can plot each one of the modes calculated.

Besides, there're two options for visualization:
- `plot_mode_2d` - plotting 2D view

- `plot_mode_3d` - plotting 3D view

### 2.2.1 Plotting 3D view

Use the command `.plot_mode_3d(mode)`.


In [13]:
mode = 5
modal.plot_mode_3d(mode)

### 2.2.2 Plotting 2D view

Use the command `.plot_mode_2d(mode)`.


In [14]:
modal.plot_mode_2d(mode)

## 2.3 Torsional analysis

You can filter modes by the mode type attribute (`"Lateral"`, `"Axial"`, `"Torsional"`), for example:

In [15]:
torsional_modes = [
    mode for mode, shape in enumerate(modal.shapes) if shape.mode_type == "Torsional"
]
torsional_modes

[6]

In [16]:
modal.plot_mode_3d(torsional_modes[0], animation=True)

# Section 3: Campbell Diagram

Also called **_whirl speed map_** in rotordynamics, ROSS calculate and plots the modes’ damped eigenvalues and the logarithmic decrement as a function of rotor speed. 

To run the Campbell Diagram, use the command `.run_campbell()`. The user must input an array of speeds, which will be iterated to calculate each point on the graph.

This method returns the damped natural frequencies, logarithmic decrement and the whirl values (values indicating the whirl direction: backward or forward).

## 3.1 Running campbell diagram

In this example the whirl speed map is calculated for a speed range from 0 to 1000 rad/s (~9550 RPM).

Storing the results, it's possible to return the following arrays:
- `wd`

- `log_dec`

- `whirl_values`

Each value in these arrays is calculated for each speed value in `speed_range`

In [17]:
samples = 31
speed_range = np.linspace(315, 1150, samples)

campbell = rotor3.run_campbell(speed_range)

/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: Extrapolating the coefficients of Bearing 0 outside its speed axis (418.9 to 1152 rad/s). Be careful when post-processing the results.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: Extrapolating the coefficients of Seal 1 outside its speed axis (418.9 to 2094 rad/s). Be careful when post-processing the results.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: Extrapolating the coefficients of Seal 2 outside its speed axis (418.9 to 2094 rad/s). Be careful when post-processing the results.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: Extrapolating the coefficients of Seal 3 outside its speed axis (418.9 to 2094 rad/s). Be careful when post-processing the results.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: E

In [18]:
# results for each frequency
frequency_index = 0
print(campbell.wd[:, frequency_index])

[947.54735609 946.66365928 945.68554756 944.60345747 943.41251002
 942.17243642 940.92460554 939.69583147 938.51754327 937.41112754
 936.38957936 935.46284285 934.65153952 933.99130004 933.47559415
 933.09213204 932.85785048 932.79030324 932.86927874 933.07334653
 933.41302578 933.88593548 934.47566356 935.16883948 935.97069214
 936.8711403  937.85652134 938.90945446 940.02534114 941.23299243
 942.56244298]


In [19]:
# results for each frequency
frequency_index = 1
print(campbell.log_dec[:, frequency_index])

[1.0713784  1.06039268 1.04894794 1.03692711 1.02419076 1.01031997
 0.99512672 0.97858624 0.96072266 0.94161772 0.92141112 0.90026115
 0.87830635 0.85563848 0.83239265 0.80869804 0.78465278 0.76035095
 0.73589827 0.71138361 0.68683205 0.66227098 0.63773953 0.61326859
 0.58887451 0.56458616 0.5404312  0.51645796 0.49267659 0.46894391
 0.4451154 ]


## 3.2 Plotting results

Now that the results are stored, use `.plot()` method to display the Campbell Diagram plot.

For the Campbell Diagram, you can plot more than one harmonic. As default, the plot display only the 1x speed option. 
Input a list with more harmonics to display it at the graph.

In [20]:
campbell.plot(harmonics=[0.5, 1])